## DC Area Report可视化工具

In [ ]:
!pip install plotly
!pip install pandas
!pip install ipywidgets

In [1]:
import re
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.offline as pyo
from IPython.display import display, HTML
import json
import time
import os
from datetime import datetime

class RPTHierarchyVisualizer:
    def __init__(self, rpt_content):
        print("开始解析RPT文件...")
        start_time = time.time()
        self.hierarchy = self.parse_rpt(rpt_content)
        parse_time = time.time() - start_time
        print(f"解析完成，耗时: {parse_time:.2f}秒")
        
    def parse_rpt(self, content):
        """优化的RPT文件解析方法"""
        hierarchy = {}
        
        # 如果content是文件路径，读取文件
        if isinstance(content, str) and content.endswith('.rpt'):
            print(f"正在读取文件: {content}")
            try:
                with open(content, 'r', encoding='utf-8') as f:
                    content = f.read()
            except UnicodeDecodeError:
                with open(content, 'r', encoding='latin-1') as f:
                    content = f.read()
        
        # 使用更高效的方式查找起始位置
        hierarchical_start = content.find("Hierarchical cell")
        if hierarchical_start == -1:
            print("未找到层次化数据区域")
            return hierarchy
            
        # 从找到的位置开始处理
        content_from_start = content[hierarchical_start:]
        lines = content_from_start.split('\n')
        
        # 找到数据开始行（跳过表头）
        start_idx = 0
        for i, line in enumerate(lines):
            if "---" in line and len(line.strip()) > 10:
                start_idx = i + 1
                break
        
        if start_idx == 0:
            print("未找到数据开始位置")
            return hierarchy
            
        # 预编译正则表达式提高性能
        pattern = re.compile(
            r'^([^\s].*?)\s+(\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)\s+(\d+\.\d+)\s+(.*)$'
        )
        
        valid_lines = []
        for line in lines[start_idx:]:
            line = line.strip()
            if not line or line.startswith('---') or len(line) < 20:
                continue
            valid_lines.append(line)
        
        print(f"找到 {len(valid_lines)} 行有效数据")
        
        # 批量处理数据 - 使用简单的文本进度条
        processed_data = []
        total_lines = len(valid_lines)
        
        for i, line in enumerate(valid_lines):
            # 简单的文本进度条
            if i % 1000 == 0 or i == total_lines - 1:
                progress = (i + 1) / total_lines * 100
                bar_length = 30
                filled_length = int(bar_length * progress // 100)
                bar = '█' * filled_length + '░' * (bar_length - filled_length)
                print(f"\r解析进度: [{bar}] {progress:.1f}% ({i+1}/{total_lines})", end='', flush=True)
            
            match = pattern.match(line)
            if match:
                try:
                    full_path = match.group(1).strip()
                    absolute_total = float(match.group(2))
                    percent_total = float(match.group(3))
                    combinational = float(match.group(4))
                    noncombinational = float(match.group(5))
                    black_boxes = float(match.group(6))
                    design = match.group(7).strip()
                    
                    processed_data.append({
                        'full_path': full_path,
                        'absolute_total': absolute_total,
                        'percent_total': percent_total,
                        'combinational': combinational,
                        'noncombinational': noncombinational,
                        'black_boxes': black_boxes,
                        'design': design
                    })
                except ValueError as e:
                    continue  # 跳过无法解析的行
        
        print(f"\n成功解析 {len(processed_data)} 条记录")
        
        # 批量构建层次结构
        print("构建层次结构...")
        hierarchy = self._build_hierarchy_fast(processed_data)
        
        return hierarchy
    
    def _build_hierarchy_fast(self, data_list):
        """快速构建层次结构"""
        hierarchy = {}
        total_items = len(data_list)
        
        for idx, data in enumerate(data_list):
            # 简单的文本进度条
            if idx % 500 == 0 or idx == total_items - 1:
                progress = (idx + 1) / total_items * 100
                bar_length = 30
                filled_length = int(bar_length * progress // 100)
                bar = '█' * filled_length + '░' * (bar_length - filled_length)
                print(f"\r构建进度: [{bar}] {progress:.1f}% ({idx+1}/{total_items})", end='', flush=True)
            
            full_path = data['full_path']
            parts = full_path.split('/')
            
            # 使用引用而不是每次遍历
            current_level = hierarchy
            
            for i, part in enumerate(parts):
                if part not in current_level:
                    current_level[part] = {
                        'name': part,
                        'full_path': '/'.join(parts[:i+1]),
                        'children': {},
                        'absolute_total': 0,
                        'percent_total': 0,
                        'combinational': 0,
                        'noncombinational': 0,
                        'black_boxes': 0,
                        'design': ''
                    }
                
                # 只在最后一层设置数据
                if i == len(parts) - 1:
                    current_level[part].update({
                        'absolute_total': data['absolute_total'],
                        'percent_total': data['percent_total'],
                        'combinational': data['combinational'],
                        'noncombinational': data['noncombinational'],
                        'black_boxes': data['black_boxes'],
                        'design': data['design']
                    })
                
                current_level = current_level[part]['children']
        
        print()  # 换行
        return hierarchy
    
    # def get_node_info(self, path):
    #     """获取指定路径节点的信息"""
    #     parts = path.split('/')
    #     current_level = self.hierarchy
    #     node = None
        
    #     for part in parts:
    #         if part in current_level:
    #             node = current_level[part]
    #             current_level = node['children']
    #         else:
    #             return None
    #     return node
    
    def generate_html_report(self, output_file="rpt_analysis_report.html"):
        """生成交互式HTML报告"""
        print(f"正在生成交互式HTML报告: {output_file}")
        
        # 序列化整个层次结构为JSON，供JavaScript使用
        print("序列化层次结构数据...")
        hierarchy_json = json.dumps(self.hierarchy, ensure_ascii=False, indent=None)
        
        # 生成HTML内容
        html_content = self._generate_interactive_html_content(hierarchy_json)
        
        # 写入文件
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f"交互式HTML报告已生成: {output_file}")
        print(f"文件大小: {os.path.getsize(output_file) / 1024:.1f} KB")
        
        # 在notebook中显示链接
        display(HTML(f'<a href="{output_file}" target="_blank" style="display: inline-block; padding: 10px 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; text-decoration: none; border-radius: 8px; font-weight: bold;">🚀 点击打开交互式HTML报告</a>'))
        
        return output_file
    
    # def _get_level_data(self, path):
    #     """获取指定路径的数据"""
    #     current_level = self.hierarchy
    #     for part in path:
    #         if part in current_level:
    #             current_level = current_level[part]['children']
    #         else:
    #             return []
        
    #     data = []
    #     for name, node in current_level.items():
    #         if node['absolute_total'] > 0:
    #             data.append({
    #                 '模块': name,
    #                 '绝对面积': node['absolute_total'],
    #                 '占比(%)': node['percent_total'],
    #                 '组合面积': node['combinational'],
    #                 '非组合面积': node['noncombinational'],
    #                 '黑盒面积': node['black_boxes'],
    #                 '设计': node['design'],
    #                 '完整路径': node['full_path']
    #             })
        
    #     # 按面积排序
    #     data.sort(key=lambda x: x['绝对面积'], reverse=True)
    #     return data
    
    # def _create_overview_chart(self, data, title):
    #     """创建总览柱状图"""
    #     df = pd.DataFrame(data)
        
    #     fig = px.bar(
    #         df, 
    #         x='模块', 
    #         y='绝对面积',
    #         hover_data=['占比(%)', '组合面积', '非组合面积', '黑盒面积', '设计'],
    #         title=title,
    #         labels={'绝对面积': 'Area (μm²)', '模块': 'Module'},
    #         color='绝对面积',
    #         color_continuous_scale='viridis'
    #     )
        
    #     fig.update_layout(
    #         height=600,
    #         showlegend=False,
    #         title_font_size=20,
    #         xaxis_title_font_size=14,
    #         yaxis_title_font_size=14,
    #         xaxis_tickangle=45
    #     )
        
    #     return fig
    
    # def _create_detailed_analysis_chart(self, data):
    #     """创建详细分析图（组合图表）"""
    #     df = pd.DataFrame(data)
        
    #     # 创建多个单独的图表，避免subplot兼容性问题
    #     figures = []
        
    #     # 1. 绝对面积柱状图
    #     fig1 = px.bar(
    #         df, x='模块', y='绝对面积', 
    #         title='绝对面积分布',
    #         color='绝对面积',
    #         color_continuous_scale='blues'
    #     )
    #     fig1.update_layout(height=400, showlegend=False)
    #     figures.append(fig1)
        
    #     # 2. 面积占比饼图
    #     fig2 = px.pie(
    #         df, values='占比(%)', names='模块',
    #         title='面积占比分布'
    #     )
    #     fig2.update_layout(height=400)
    #     figures.append(fig2)
        
    #     # 3. 组合vs非组合面积对比
    #     comparison_data = []
    #     for _, row in df.iterrows():
    #         comparison_data.extend([
    #             {'模块': row['模块'], '面积类型': '组合面积', '面积': row['组合面积']},
    #             {'模块': row['模块'], '面积类型': '非组合面积', '面积': row['非组合面积']}
    #         ])
        
    #     comp_df = pd.DataFrame(comparison_data)
    #     fig3 = px.bar(
    #         comp_df, x='模块', y='面积', color='面积类型',
    #         title='组合 vs 非组合面积对比',
    #         color_discrete_map={'组合面积': 'lightgreen', '非组合面积': 'lightcoral'}
    #     )
    #     fig3.update_layout(height=400)
    #     figures.append(fig3)
        
    #     # 4. 黑盒面积趋势
    #     fig4 = px.scatter(
    #         df, x='模块', y='黑盒面积',
    #         title='黑盒面积分布',
    #         color='黑盒面积',
    #         color_continuous_scale='oranges'
    #     )
    #     fig4.update_traces(mode='markers+lines')
    #     fig4.update_layout(height=400)
    #     figures.append(fig4)
        
    #     # 合并所有图表为一个HTML div
    #     return figures
    
    # def _create_area_type_distribution(self, data):
    #     """创建面积类型分布饼图"""
    #     df = pd.DataFrame(data)
        
    #     # 计算总和
    #     total_comb = df['组合面积'].sum()
    #     total_non_comb = df['非组合面积'].sum()
    #     total_black = df['黑盒面积'].sum()
        
    #     labels = ['组合逻辑面积', '非组合逻辑面积', '黑盒面积']
    #     values = [total_comb, total_non_comb, total_black]
    #     colors = ['lightblue', 'lightgreen', 'orange']
        
    #     fig = go.Figure(data=[go.Pie(
    #         labels=labels, 
    #         values=values,
    #         hole=0.3,
    #         marker_colors=colors,
    #         textinfo='label+percent+value',
    #         textfont_size=12
    #     )])
        
    #     fig.update_layout(
    #         title="总体面积类型分布",
    #         title_font_size=20,
    #         height=500
    #     )
        
    #     return fig
    
    # def _create_hierarchy_treemap(self):
    #     """创建层次结构树状图"""
    #     # 收集所有节点数据
    #     all_nodes = []
        
    #     def collect_nodes(level, parent_path=""):
    #         for name, node in level.items():
    #             if node['absolute_total'] > 0:
    #                 current_path = f"{parent_path}/{name}" if parent_path else name
    #                 all_nodes.append({
    #                     'id': current_path,
    #                     'parent': parent_path if parent_path else "",
    #                     'value': node['absolute_total'],
    #                     'label': name
    #                 })
    #                 if node['children']:
    #                     collect_nodes(node['children'], current_path)
        
    #     collect_nodes(self.hierarchy)
        
    #     if not all_nodes:
    #         return go.Figure()
        
    #     df = pd.DataFrame(all_nodes)
        
    #     fig = px.treemap(
    #         df,
    #         ids='id',
    #         parents='parent',
    #         values='value',
    #         names='label',
    #         title="层次结构树状图"
    #     )
        
    #     fig.update_layout(
    #         title_font_size=20,
    #         height=600
    #     )
        
    #     return fig
    
    def _generate_interactive_html_content(self, hierarchy_json):
        """生成完整的交互式HTML内容，包含多张图表"""
        html_template = f"""
        <!DOCTYPE html>
        <html lang="zh-CN">
        <head>
            <meta charset="UTF-8">
            <meta name="viewport" content="width=device-width, initial-scale=1.0">
            <title>RPT 交互式分析报告</title>
            <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
            <style>
                body {{
                    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                    margin: 0;
                    padding: 20px;
                    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    min-height: 100vh;
                }}
                .container {{
                    max-width: 1400px;
                    margin: 0 auto;
                    background: white;
                    border-radius: 15px;
                    box-shadow: 0 10px 30px rgba(0,0,0,0.2);
                    overflow: hidden;
                }}
                .header {{
                    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    color: white;
                    padding: 30px;
                    text-align: center;
                }}
                .header h1 {{
                    margin: 0;
                    font-size: 2.5em;
                    text-shadow: 2px 2px 4px rgba(0,0,0,0.3);
                }}
                .header p {{
                    margin: 10px 0 0 0;
                    opacity: 0.9;
                }}
                .content {{
                    padding: 30px;
                }}
                .breadcrumb {{
                    background: #f8f9fa;
                    padding: 15px;
                    border-radius: 8px;
                    margin-bottom: 20px;
                    border-left: 4px solid #667eea;
                }}
                .breadcrumb-item {{
                    display: inline-block;
                    padding: 5px 10px;
                    margin: 2px;
                    background: white;
                    border-radius: 4px;
                    cursor: pointer;
                    transition: all 0.3s ease;
                    text-decoration: none;
                    color: #333;
                }}
                .breadcrumb-item:hover {{
                    background: #e9ecef;
                    transform: translateY(-1px);
                    box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                }}
                .summary-box {{
                    background: #f8f9fa;
                    border-radius: 10px;
                    padding: 20px;
                    margin-bottom: 30px;
                    border-left: 5px solid #667eea;
                }}
                .stats-grid {{
                    display: grid;
                    grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                    gap: 20px;
                    margin-top: 20px;
                }}
                .stat-item {{
                    text-align: center;
                    padding: 15px;
                    background: white;
                    border-radius: 8px;
                    box-shadow: 0 2px 10px rgba(0,0,0,0.1);
                }}
                .stat-item h3 {{
                    margin: 0 0 10px 0;
                    color: #667eea;
                    font-size: 1em;
                }}
                .stat-item p {{
                    margin: 0;
                    font-size: 1.2em;
                    font-weight: bold;
                    color: #333;
                }}
                .chart-container {{
                    margin-bottom: 40px;
                    background: linear-gradient(145deg, #ffffff, #f8f9fa);
                    border-radius: 15px;
                    padding: 25px;
                    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.1);
                    border: 1px solid rgba(102, 126, 234, 0.1);
                    transition: all 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275);
                }}
                .chart-container:hover {{
                    transform: translateY(-5px);
                    box-shadow: 0 15px 35px rgba(102, 126, 234, 0.15);
                    border-color: rgba(102, 126, 234, 0.3);
                }}
                .chart-container h2 {{
                    margin-top: 0;
                    color: #667eea;
                    padding-bottom: 20px;
                    border-bottom: 3px solid;
                    border-image: linear-gradient(90deg, #667eea, #764ba2) 1;
                    font-size: 1.5em;
                    font-weight: 600;
                    position: relative;
                    display: flex;
                    justify-content: space-between;
                    align-items: center;
                }}
                .chart-title {{
                    flex: 1;
                    text-align: left;
                }}
                .charts-row {{
                    display: flex;
                    gap: 25px;
                    margin-bottom: 40px;
                }}
                .chart-proportion {{
                    flex: 6;
                }}
                .chart-type {{
                    flex: 4;
                }}
                .chart-item {{
                    background: linear-gradient(145deg, #ffffff, #f8f9fa);
                    border-radius: 15px;
                    padding: 25px;
                    box-shadow: 0 8px 25px rgba(102, 126, 234, 0.1);
                    transition: all 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275);
                    border: 1px solid rgba(102, 126, 234, 0.1);
                }}
                .chart-item:hover {{
                    transform: translateY(-8px) scale(1.02);
                    box-shadow: 0 20px 40px rgba(102, 126, 234, 0.2);
                    border-color: rgba(102, 126, 234, 0.3);
                }}
                .chart-item h3 {{
                    margin-top: 0;
                    color: #667eea;
                    text-align: center;
                    padding-bottom: 15px;
                    border-bottom: 3px solid;
                    border-image: linear-gradient(90deg, #667eea, #764ba2) 1;
                    font-weight: 600;
                    font-size: 1.1em;
                    position: relative;
                }}
                .chart-toggle {{
                    display: flex;
                    gap: 8px;
                    margin: 0;
                    flex-shrink: 0;
                }}
                .toggle-btn {{
                    padding: 6px 12px;
                    border: 2px solid #667eea;
                    background: white;
                    color: #667eea;
                    border-radius: 20px;
                    cursor: pointer;
                    transition: all 0.3s ease;
                    font-size: 0.85em;
                    font-weight: 500;
                }}
                .toggle-btn:hover {{
                    background: #667eea;
                    color: white;
                    transform: translateY(-2px);
                }}
                .toggle-btn.active {{
                    background: #667eea;
                    color: white;
                    box-shadow: 0 4px 8px rgba(102, 126, 234, 0.3);
                }}
                .controls {{
                    text-align: center;
                    margin-bottom: 20px;
                }}
                .controls-row {{
                    display: flex;
                    justify-content: center;
                    align-items: center;
                    gap: 15px;
                    margin-bottom: 10px;
                }}
                .btn {{
                    display: inline-block;
                    padding: 10px 20px;
                    margin: 5px;
                    background: #667eea;
                    color: white;
                    border: none;
                    border-radius: 5px;
                    cursor: pointer;
                    transition: all 0.3s ease;
                    font-size: 14px;
                }}
                .btn:hover:not(:disabled) {{
                    background: #5a67d8;
                    transform: translateY(-1px);
                    box-shadow: 0 4px 8px rgba(0,0,0,0.2);
                }}
                .btn:disabled {{
                    background: #ccc;
                    cursor: not-allowed;
                    transform: none;
                    box-shadow: none;
                }}
                .unit-selector {{
                    padding: 8px 12px;
                    border: 2px solid #667eea;
                    border-radius: 5px;
                    background: white;
                    font-size: 14px;
                    cursor: pointer;
                }}
                .info-box {{
                    background: #e3f2fd;
                    border: 1px solid #2196f3;
                    border-radius: 8px;
                    padding: 15px;
                    margin: 20px 0;
                }}
                .footer {{
                    text-align: center;
                    padding: 20px;
                    color: #666;
                    background: #f8f9fa;
                    border-top: 1px solid #eee;
                }}
                @media (max-width: 1024px) {{
                    .charts-row {{
                        flex-direction: column;
                    }}
                    .chart-proportion, .chart-type {{
                        flex: none;
                    }}
                }}
                @media (max-width: 768px) {{
                    .controls-row {{
                        flex-direction: column;
                    }}
                    .chart-container {{
                        padding: 20px;
                    }}
                    .chart-container:hover {{
                        transform: translateY(-3px);
                    }}
                    .chart-toggle {{
                        margin-top: 10px;
                        margin-bottom: 5px;
                    }}
                    .toggle-btn {{
                        padding: 5px 10px;
                        font-size: 0.8em;
                    }}
                }}
            </style>
        </head>
        <body>
            <div class="container">
                <div class="header">
                    <h1>🚀 RPT 交互式分析报告</h1>
                    <p>生成时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                    <p>💡 点击图表元素可以向下钻取探索</p>
                </div>
                <div class="content">
                    <div class="info-box">
                        <strong>🎯 使用说明:</strong> 
                        点击任意图表的模块可以查看其子模块。使用面包屑导航可以快速返回上级目录。
                    </div>
                    
                    <div class="breadcrumb">
                        <strong>📍 当前位置:</strong> <span id="breadcrumb-content">根目录</span>
                    </div>
                    
                    <div class="controls">
                        <div class="controls-row">
                            <button class="btn" onclick="goBack()" id="backBtn" disabled>⬅️ 返回上级</button>
                            <button class="btn" onclick="goHome()">🏠 返回根目录</button>
                            <select class="unit-selector" id="unitSelector" onchange="changeUnit()">
                                <option value="μm²">μm²</option>
                                <option value="mm²">mm²</option>
                                <option value="K">K μm²</option>
                                <option value="M">M μm²</option>
                            </select>
                        </div>
                    </div>
                    
                    <div id="summary-section"></div>
                    
                    <div class="charts-row">
                        <div class="chart-container chart-proportion">
                            <h2>
                                <span class="chart-title">📊 面积占比</span>
                                <div class="chart-toggle">
                                    <button class="toggle-btn active" id="pie-btn" onclick="switchChart('pie')">🥧 饼图</button>
                                    <button class="toggle-btn" id="bar-btn" onclick="switchChart('bar')">📊 柱状图</button>
                                </div>
                            </h2>
                            <div id="proportion-chart"></div>
                        </div>
                        
                        <div class="chart-container chart-type">
                            <h2>🔍 面积类型分布</h2>
                            <div id="type-chart"></div>
                        </div>
                    </div>
                    
                    <div class="chart-container">
                        <h2>🌳 层次结构树状图</h2>
                        <div id="treemap-chart"></div>
                    </div>
                    

                </div>
                <div class="footer">
                    <p>📈 本报告由 RPT 层次化分析工具自动生成 | 🎯 交互式版本</p>
                </div>
            </div>

            <script>
                // 全局变量
                const hierarchyData = {hierarchy_json};
                let currentPath = [];
                let currentUnit = 'μm²';
                let unitMultiplier = 1;
                let currentChartType = 'pie'; // 'pie' 或 'bar'

                // 单位转换工具函数
                function updateUnitMultiplier() {{
                    switch(currentUnit) {{
                        case 'μm²': unitMultiplier = 1; break;
                        case 'mm²': unitMultiplier = 1e-6; break;
                        case 'K': unitMultiplier = 1e-3; break;
                        case 'M': unitMultiplier = 1e-6; break;
                        default: unitMultiplier = 1;
                    }}
                }}

                function convertArea(area) {{
                    return area * unitMultiplier;
                }}

                function formatArea(area) {{
                    const converted = convertArea(area);
                    if (currentUnit === 'K' || currentUnit === 'M') {{
                        return converted.toFixed(2);
                    }} else if (converted >= 1000) {{
                        return converted.toFixed(0);
                    }} else {{
                        return converted.toFixed(3);
                    }}
                }}

                function getUnitLabel() {{
                    return currentUnit;
                }}

                function changeUnit() {{
                    const selector = document.getElementById('unitSelector');
                    currentUnit = selector.value;
                    updateUnitMultiplier();
                    updateDisplay();
                }}

                // 初始化
                document.addEventListener('DOMContentLoaded', function() {{
                    console.log('页面加载完成，开始初始化...');
                    console.log('层次数据:', hierarchyData);
                    updateUnitMultiplier();
                    updateDisplay();
                }});

                // 获取当前路径的数据
                function getCurrentLevelData() {{
                    let current = hierarchyData;
                    for (let part of currentPath) {{
                        if (current[part] && current[part].children) {{
                            current = current[part].children;
                        }} else {{
                            console.warn('路径不存在:', currentPath, part);
                            return {{}};
                        }}
                    }}
                    console.log('当前层级数据:', current);
                    return current;
                }}

                // 更新显示
                function updateDisplay() {{
                    console.log('更新显示，当前路径:', currentPath);
                    updateBreadcrumb();
                    updateSummary();
                    updateAllCharts();
                    updateControls();
                }}

                // 更新面包屑导航（修复点击功能）
                function updateBreadcrumb() {{
                    const breadcrumb = document.getElementById('breadcrumb-content');
                    let html = '<span class="breadcrumb-item" onclick="navigateTo([])">根目录</span>';
                    
                    for (let i = 0; i < currentPath.length; i++) {{
                        const path = currentPath.slice(0, i + 1);
                        const pathStr = JSON.stringify(path).replace(/"/g, '&quot;');
                        html += ` > <span class="breadcrumb-item" onclick='navigateTo(${{pathStr}})'>${{currentPath[i]}}</span>`;
                    }}
                    
                    breadcrumb.innerHTML = html;
                }}

                // 更新统计摘要
                function updateSummary() {{
                    const data = getCurrentLevelData();
                    const modules = Object.keys(data).filter(key => data[key] && data[key].absolute_total > 0);
                    
                    console.log('找到模块:', modules);
                    
                    if (modules.length === 0) {{
                        document.getElementById('summary-section').innerHTML = '<div class="summary-box"><h2>📊 统计摘要</h2><p>没有可用的子模块数据</p></div>';
                        return;
                    }}

                    let totalArea = 0;
                    let totalComb = 0;
                    let totalNonComb = 0;
                    let maxModule = null;
                    let maxArea = 0;

                    modules.forEach(name => {{
                        const node = data[name];
                        totalArea += node.absolute_total;
                        totalComb += node.combinational;
                        totalNonComb += node.noncombinational;
                        
                        if (node.absolute_total > maxArea) {{
                            maxArea = node.absolute_total;
                            maxModule = name;
                        }}
                    }});

                    const pathDisplay = currentPath.length > 0 ? '/' + currentPath.join('/') : '根目录';
                    const unitLabel = getUnitLabel();
                    
                    document.getElementById('summary-section').innerHTML = `
                        <div class="summary-box">
                            <h2>📊 统计摘要 - ${{pathDisplay}}</h2>
                            <div class="stats-grid">
                                <div class="stat-item">
                                    <h3>子模块数量</h3>
                                    <p>${{modules.length}}</p>
                                </div>
                                <div class="stat-item">
                                    <h3>总面积</h3>
                                    <p>${{formatArea(totalArea)}} ${{unitLabel}}</p>
                                </div>
                                <div class="stat-item">
                                    <h3>组合逻辑面积</h3>
                                    <p>${{formatArea(totalComb)}} ${{unitLabel}}</p>
                                </div>
                                <div class="stat-item">
                                    <h3>最大模块</h3>
                                    <p>${{maxModule}}<br>(${{formatArea(maxArea)}} ${{unitLabel}})</p>
                                </div>
                            </div>
                        </div>
                    `;
                }}

                // 更新所有图表
                function updateAllCharts() {{
                    const data = getCurrentLevelData();
                    const modules = Object.keys(data).filter(key => data[key] && data[key].absolute_total > 0);
                    
                    if (modules.length === 0) {{
                        ['proportion-chart', 'type-chart', 'treemap-chart'].forEach(id => {{
                            const element = document.getElementById(id);
                            if (element) {{
                                element.innerHTML = '<p style="text-align: center; color: #666; padding: 50px;">没有可用数据</p>';
                            }}
                        }});
                        return;
                    }}

                    // 准备数据
                    const chartData = modules.map(name => ({{
                        name: name,
                        area: data[name].absolute_total,
                        percent: data[name].percent_total,
                        combinational: data[name].combinational,
                        noncombinational: data[name].noncombinational,
                        black_boxes: data[name].black_boxes,
                        design: data[name].design
                    }})).sort((a, b) => b.area - a.area);

                    console.log('图表数据:', chartData);

                    createProportionChart(chartData);
                    createTypeChart(chartData);
                    createTreemapChart();
                }}

                // 创建面积占比图表（支持饼图/柱状图切换）
                function createProportionChart(data) {{
                    if (currentChartType === 'pie') {{
                        createPieChart(data);
                    }} else {{
                        createBarChart(data);
                    }}
                }}

                // 创建饼图
                function createPieChart(data) {{
                    const unitLabel = getUnitLabel();
                    
                    // 生成初始颜色
                    const originalColors = data.map((d, i) => `hsl(${{200 + i * 20}}, 65%, 55%)`);
                    const originalPull = data.map(() => 0.02);
                    
                    const trace = {{
                        labels: data.map(d => d.name),
                        values: data.map(d => convertArea(d.area)),
                        type: 'pie',
                        hovertemplate: '<b>%{{label}}</b><br>面积: %{{value:,.3f}} ' + unitLabel + '<br>占比: %{{percent}}<extra></extra>',
                        textinfo: 'label+percent',
                        textposition: 'auto',
                        hole: 0.35,
                        pull: originalPull,
                        marker: {{
                            colors: originalColors,
                            line: {{
                                color: '#FFFFFF',
                                width: 3
                            }}
                        }},
                        textfont: {{
                            size: 11,
                            color: 'white'
                        }},
                        sort: false  // 保持原始顺序，避免重排序导致的问题
                    }};

                    const layout = {{
                        margin: {{ t: 20, b: 20, l: 20, r: 20 }},
                        height: 500,
                        paper_bgcolor: 'rgba(0,0,0,0)',
                        showlegend: true,
                        legend: {{
                            orientation: 'v',
                            x: 1.02,
                            y: 0.5
                        }}
                    }};

                    const config = {{
                        displayModeBar: true,
                        displaylogo: false,
                        responsive: true
                    }};

                    // 清除之前的事件监听器
                    const chartElement = document.getElementById('proportion-chart');
                    if (chartElement._plotlyInitialized) {{
                        Plotly.purge('proportion-chart');
                    }}

                    Plotly.newPlot('proportion-chart', [trace], layout, config).then(() => {{
                        chartElement._plotlyInitialized = true;
                        
                        // 添加点击事件
                        chartElement.on('plotly_click', function(eventData) {{
                            if (eventData.points && eventData.points[0]) {{
                                const moduleName = eventData.points[0].label;
                                drillDown(moduleName);
                            }}
                        }});

                        // 修复悬停动态效果
                        chartElement.on('plotly_hover', function(eventData) {{
                            if (eventData.points && eventData.points[0]) {{
                                const pointIndex = eventData.points[0].pointIndex;
                                
                                // 创建新的数组，确保每次都是全新的引用
                                const newPull = data.map((d, i) => i === pointIndex ? 0.25 : 0.02);
                                const newColors = data.map((d, i) => i === pointIndex ? '#ff6b6b' : originalColors[i]);
                                const newLineWidth = data.map((d, i) => i === pointIndex ? 6 : 3);
                                
                                const updates = {{
                                    pull: [newPull],
                                    'marker.colors': [newColors],
                                    'marker.line.width': [newLineWidth]
                                }};
                                
                                Plotly.restyle('proportion-chart', updates, [0]);
                            }}
                        }});

                        chartElement.on('plotly_unhover', function(eventData) {{
                            // 重置到原始状态
                            const resetUpdates = {{
                                pull: [originalPull],
                                'marker.colors': [originalColors],
                                'marker.line.width': [data.map(() => 3)]
                            }};
                            Plotly.restyle('proportion-chart', resetUpdates, [0]);
                        }});
                    }});
                }}

                // 创建柱状图
                function createBarChart(data) {{
                    const unitLabel = getUnitLabel();
                    
                    // 生成初始颜色和状态
                    const originalColors = data.map((d, i) => `hsl(${{200 + i * 20}}, 65%, 55%)`);
                    const originalY = data.map(d => convertArea(d.area));
                    
                    const trace = {{
                        x: data.map(d => d.name),
                        y: originalY,
                        type: 'bar',
                        marker: {{
                            color: originalColors,
                            opacity: 0.8,
                            line: {{
                                color: 'rgba(255,255,255,0.4)',
                                width: 2
                            }}
                        }},
                        hovertemplate: '<b>%{{x}}</b><br>面积: %{{y:,.3f}} ' + unitLabel + '<br>占比: %{{customdata[0]:.2f}}%<br>设计: %{{customdata[1]}}<extra></extra>',
                        customdata: data.map(d => [d.percent, d.design])
                    }};

                    const layout = {{
                        margin: {{ t: 20, b: 60, l: 60, r: 20 }},
                        height: 500,
                        xaxis: {{ 
                            title: 'Module',
                            tickangle: -45,
                            tickfont: {{ size: 10 }}
                        }},
                        yaxis: {{ title: 'Area (' + unitLabel + ')' }},
                        hovermode: 'closest',
                        plot_bgcolor: 'rgba(0,0,0,0)',
                        paper_bgcolor: 'rgba(0,0,0,0)'
                    }};

                    const config = {{
                        displayModeBar: true,
                        displaylogo: false,
                        modeBarButtonsToRemove: ['pan2d', 'lasso2d'],
                        responsive: true
                    }};

                    // 清除之前的事件监听器
                    const chartElement = document.getElementById('proportion-chart');
                    if (chartElement._plotlyInitialized) {{
                        Plotly.purge('proportion-chart');
                    }}

                    Plotly.newPlot('proportion-chart', [trace], layout, config).then(() => {{
                        chartElement._plotlyInitialized = true;
                        
                        // 添加点击事件
                        chartElement.on('plotly_click', function(eventData) {{
                            if (eventData.points && eventData.points[0]) {{
                                const moduleName = eventData.points[0].x;
                                drillDown(moduleName);
                            }}
                        }});

                        // 增强悬停动态效果 - 柱子放大和高亮
                        chartElement.on('plotly_hover', function(eventData) {{
                            if (eventData.points && eventData.points[0]) {{
                                const pointIndex = eventData.points[0].pointNumber;
                                
                                // 放大悬停的柱子，缩小其他柱子
                                const scaledY = originalY.map((val, i) => i === pointIndex ? val * 1.15 : val * 0.95);
                                const newColors = data.map((d, i) => i === pointIndex ? '#ff6b6b' : originalColors[i]);
                                const newOpacity = data.map((d, i) => i === pointIndex ? 1.0 : 0.6);
                                const newLineWidth = data.map((d, i) => i === pointIndex ? 4 : 2);
                                const newLineColor = data.map((d, i) => i === pointIndex ? 'rgba(255,255,255,0.9)' : 'rgba(255,255,255,0.4)');
                                
                                const updates = {{
                                    y: [scaledY],
                                    'marker.opacity': [newOpacity],
                                    'marker.line.width': [newLineWidth],
                                    'marker.line.color': [newLineColor],
                                    'marker.color': [newColors]
                                }};
                                Plotly.restyle('proportion-chart', updates, [0]);
                            }}
                        }});

                        chartElement.on('plotly_unhover', function(eventData) {{
                            // 重置到原始状态
                            const resetUpdates = {{
                                y: [originalY],
                                'marker.opacity': [data.map(() => 0.8)],
                                'marker.line.width': [data.map(() => 2)],
                                'marker.line.color': [data.map(() => 'rgba(255,255,255,0.4)')],
                                'marker.color': [originalColors]
                            }};
                            Plotly.restyle('proportion-chart', resetUpdates, [0]);
                        }});
                    }});
                }}

                // 图表切换函数
                function switchChart(type) {{
                    currentChartType = type;
                    
                    // 更新按钮状态
                    document.getElementById('pie-btn').classList.toggle('active', type === 'pie');
                    document.getElementById('bar-btn').classList.toggle('active', type === 'bar');
                    
                    // 重新创建图表
                    updateAllCharts();
                }}

                // 创建面积类型分布图
                function createTypeChart(data) {{
                    const totalComb = data.reduce((sum, d) => sum + d.combinational, 0);
                    const totalNonComb = data.reduce((sum, d) => sum + d.noncombinational, 0);
                    const totalBlack = data.reduce((sum, d) => sum + d.black_boxes, 0);

                    const unitLabel = getUnitLabel();
                    const trace = {{
                        labels: ['组合逻辑', '非组合逻辑', '黑盒'],
                        values: [convertArea(totalComb), convertArea(totalNonComb), convertArea(totalBlack)],
                        type: 'pie',
                        hole: 0.4,
                        marker: {{
                            colors: ['lightblue', 'lightgreen', 'orange'],
                            line: {{
                                color: '#FFFFFF',
                                width: 2
                            }}
                        }},
                        textinfo: 'label+percent+value',
                        hovertemplate: '<b>%{{label}}</b><br>面积: %{{value:,.3f}} ' + unitLabel + '<br>占比: %{{percent}}<extra></extra>'
                    }};

                    const layout = {{
                        margin: {{ t: 20, b: 20, l: 20, r: 20 }},
                        height: 450,
                        paper_bgcolor: 'rgba(0,0,0,0)'
                    }};

                    const config = {{
                        displayModeBar: true,
                        displaylogo: false
                    }};

                    Plotly.newPlot('type-chart', [trace], layout, config);
                }}



                // 创建树状图（修复显示问题）
                function createTreemapChart() {{
                    try {{
                        const allNodes = [];
                        
                        function collectNodes(level, parentPath = "") {{
                            for (const [name, node] of Object.entries(level)) {{
                                if (node && typeof node === 'object' && node.absolute_total > 0) {{
                                    const currentPath = parentPath ? `${{parentPath}}/${{name}}` : name;
                                    allNodes.push({{
                                        ids: currentPath,
                                        labels: name,
                                        parents: parentPath,
                                        values: node.absolute_total,
                                        text: `${{name}}<br>${{formatArea(node.absolute_total)}} ${{getUnitLabel()}}`
                                    }});
                                    
                                    if (node.children && Object.keys(node.children).length > 0) {{
                                        collectNodes(node.children, currentPath);
                                    }}
                                }}
                            }}
                        }}
                        
                        collectNodes(hierarchyData);
                        
                        if (allNodes.length === 0) {{
                            document.getElementById('treemap-chart').innerHTML = '<p style="text-align: center; color: #666; padding: 50px;">没有层次结构数据</p>';
                            return;
                        }}

                        console.log('树状图节点数据:', allNodes);

                        const trace = {{
                            type: 'treemap',
                            ids: allNodes.map(n => n.ids),
                            labels: allNodes.map(n => n.labels),
                            parents: allNodes.map(n => n.parents),
                            values: allNodes.map(n => convertArea(n.values)),
                            text: allNodes.map(n => n.text),
                            textinfo: 'label+text',
                            hovertemplate: '<b>%{{label}}</b><br>面积: %{{value:,.3f}} ' + getUnitLabel() + '<br>路径: %{{id}}<extra></extra>',
                            branchvalues: 'total',
                            maxdepth: 3,
                            textfont: {{
                                size: 12,
                                color: 'white'
                            }},
                            marker: {{
                                colorscale: [
                                    [0, '#4a90e2'],
                                    [0.25, '#357abd'], 
                                    [0.5, '#2e5b95'],
                                    [0.75, '#667eea'],
                                    [1, '#5a7de8']
                                ],
                                line: {{
                                    color: 'white',
                                    width: 2
                                }}
                            }}
                        }};

                        const layout = {{
                            margin: {{ t: 20, b: 20, l: 20, r: 20 }},
                            height: 600,
                            paper_bgcolor: 'rgba(0,0,0,0)',
                            font: {{
                                size: 12
                            }}
                        }};

                        const config = {{
                            displayModeBar: true,
                            displaylogo: false,
                            modeBarButtonsToRemove: ['pan2d', 'lasso2d']
                        }};

                        Plotly.newPlot('treemap-chart', [trace], layout, config);
                        
                        document.getElementById('treemap-chart').on('plotly_click', function(eventData) {{
                            if (eventData.points && eventData.points[0]) {{
                                const clickedPath = eventData.points[0].id;
                                if (clickedPath) {{
                                    const pathArray = clickedPath.split('/');
                                    navigateTo(pathArray);
                                }}
                            }}
                        }});
                    }} catch (error) {{
                        console.error('创建树状图时出错:', error);
                        document.getElementById('treemap-chart').innerHTML = '<p style="text-align: center; color: #666; padding: 50px;">树状图创建失败: ' + error.message + '</p>';
                    }}
                }}

                // 钻取到下一级
                function drillDown(moduleName) {{
                    const newPath = [...currentPath, moduleName];
                    
                    // 检查是否有子模块
                    let current = hierarchyData;
                    for (let part of newPath) {{
                        if (current[part] && current[part].children) {{
                            current = current[part].children;
                        }} else {{
                            console.log('该模块没有子模块数据:', moduleName);
                            return;
                        }}
                    }}

                    const hasChildren = Object.keys(current).some(key => current[key] && current[key].absolute_total > 0);
                    if (!hasChildren) {{
                        console.log('该模块没有有效的子模块数据:', moduleName);
                        return;
                    }}

                    navigateTo(newPath);
                }}

                // 导航到指定路径（修复功能）
                function navigateTo(path) {{
                    console.log('导航到路径:', path);
                    currentPath = [...path];
                    updateDisplay();
                }}

                // 返回上级（修复功能）
                function goBack() {{
                    if (currentPath.length > 0) {{
                        currentPath.pop();
                        console.log('返回到路径:', currentPath);
                        updateDisplay();
                    }}
                }}

                // 返回根目录
                function goHome() {{
                    currentPath = [];
                    console.log('返回根目录');
                    updateDisplay();
                }}

                // 更新控制按钮状态（修复功能）
                function updateControls() {{
                    const backBtn = document.getElementById('backBtn');
                    backBtn.disabled = currentPath.length === 0;
                    
                    // 更新单位选择器
                    const unitSelector = document.getElementById('unitSelector');
                    unitSelector.value = currentUnit;
                }}
            </script>
        </body>
        </html>
        """
        
        return html_template

def generate_html_report_from_rpt(rpt_file_path, output_file="rpt_analysis_report.html"):
    """直接从RPT文件生成HTML报告"""
    print("🚀 开始生成HTML报告...")
    
    # 创建可视化器
    visualizer = RPTHierarchyVisualizer(rpt_file_path)
    
    # 生成HTML报告
    output_path = visualizer.generate_html_report(output_file)
    
    print("✅ HTML报告生成完成！")
    return output_path
  

In [ ]:
# 直接生成完整的多图表HTML报告
rpt_file_path = "../output/dc/reports/area.rpt"

# 验证文件是否存在
import os
if os.path.exists(rpt_file_path):
    print(f"文件存在: {rpt_file_path}")
    print("🔄 重新生成完整的多图表HTML报告...")
    
    # 直接生成HTML报告（包含树状图和多图表布局）
    html_file = generate_html_report_from_rpt(rpt_file_path, "../output/dc/reports/area_report.html")
    print(f"✨ 完整的多图表HTML报告已保存到: {html_file}")
    print("📊 包含的图表:")
    print("   - 📊 模块面积分布柱状图（可点击钻取）")
    print("   - 🥧 面积占比饼图（可点击钻取）") 
    print("   - 🔍 面积类型分布图")
    print("   - 🌳 层次结构树状图（可点击导航）")
    
else:
    print(f"文件不存在: {rpt_file_path}")
    print("请确认文件路径是否正确")
    
    # 列出当前目录下的.rpt文件
    rpt_files = [f for f in os.listdir('.') if f.endswith('.rpt')]
    if rpt_files:
        print(f"当前目录下的RPT文件: {rpt_files}")
        # 如果找到RPT文件，使用第一个
        html_file = generate_html_report_from_rpt(rpt_files[0], f"{rpt_files[0]}_report.html")
        print(f"✨ 完整的多图表HTML报告已保存到: {html_file}")
    else:
        print("当前目录下没有找到.rpt文件")

In [3]:
# # 在notebook中直接显示HTML报告
# html_file_path = "./area_report.html"

# try:
#     # 读取HTML文件内容
#     with open(html_file_path, 'r', encoding='utf-8') as f:
#         html_content = f.read()
    
#     # 在notebook中显示HTML
#     print(f"📊 正在显示交互式HTML报告: {html_file_path}")
#     display(HTML(html_content))
    
# except FileNotFoundError:
#     print(f"❌ 未找到HTML文件: {html_file_path}")
#     print("请先运行上面的cell生成HTML报告")
# except Exception as e:
#     print(f"❌ 显示HTML时出错: {e}")
    
#     # 如果直接显示失败，提供备选方案
#     print(f"💡 你也可以直接在浏览器中打开文件: {html_file_path}")
#     display(HTML(f'<a href="{html_file_path}" target="_blank" style="display: inline-block; padding: 10px 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; text-decoration: none; border-radius: 8px; font-weight: bold;">🚀 点击打开HTML报告</a>'))
